# Brain snRNAseq Experiment

# Overview

This notebook is the canonical ingest pipeline for the brain snRNA-seq project after ambient correction has already been completed in notebook 00. It starts from scAR-corrected per-sample `.h5ad` files, performs QC, SCVI/SOLO doublet filtering, sample concatenation via sc.concat, MapMyCells annotation, and downstream visualization/export.

## Contract with Notebook 00
- Notebook 00 is responsible for ambient correction and generation of per-sample scAR-corrected `.h5ad` files.
- This notebook expects corrected inputs at `Analysis/scAR/h5ad/*_scar_corrected.h5ad`.

## Key Steps
1. **Data Ingestion**: Load scAR-corrected `.h5ad` files for each sample.
2. **Preprocessing**: Filter cells/genes, detect doublets with scvi/SOLO, normalize, and log-transform.
3. **Integration**: Concatenate per-sample AnnData objects with sc.concat.
4. **Cell Typing**: Annotate cells using the Allen Institute's MapMyCells (whole mouse brain atlas).
5. **Analysis and Exports**: Generate class UMAP, export barcode tables, and save an annotated HVG-filtered AnnData artifact.

## Data Sources
- Samples: Mock-1/2/3 (uninfected), OG-1/2/3 (infected with *C. albicans*).
- File format: scAR-corrected `.h5ad` files in `Analysis/scAR/h5ad`.
- Cell atlas: Allen Institute MapMyCells (taxonomy: CCN20230722_SUPT, markers: mouse_markers_230821.json).

## Output Files (all tagged with source notebook)
- **Processed AnnData**: `Analysis/adatas/adata_scar_nb01_integrated.h5ad`.
- **MapMyCells Query AnnData**: `Analysis/adatas/adata_scar_nb01_geneids_query.h5ad`.
- **Class-filtered AnnData** (for class UMAP cache): `Analysis/adatas/adata_scar_nb01_class_filtered.h5ad`.
- **Annotated HVG-filtered AnnData**: `Analysis/adatas/adata_scar_nb01_annotated_hvg.h5ad`.
- **Mapping Outputs**: `Analysis/Mapping/mapping_output_nb01.csv` and `mapping_output_nb01.json`.
- **UMAP Plot**: `Analysis/Results/UMAP/UMAP_Cell_Class_nb01.svg`.
- **Barcode Tables**: `Analysis/Results/Tables/all_cell_barcodes_nb01.csv`, `Analysis/Microglia_analysis/microglia_cell_barcodes_nb01.csv`, `Analysis/Astrocyte_analysis/astrocyte_cell_barcodes_nb01.csv`.
- **Sample Table**: `Analysis/Notebooks/sample_paths_nb01.csv`.

## Self-Contained Execution
This notebook can be run independently on a fresh kernel after notebook 00 has produced the corrected inputs. It loads data from `Analysis/scAR/h5ad/` and saves outputs to `Analysis/` subdirectories.

## Runtime Estimates
- Per sample preprocessing: ~5-10 minutes (depends on hardware).
- Full pipeline: ~1-2 hours (including MapMyCells mapping, which is CPU-intensive).

## Dependencies
- Python 3.8+
- Key packages: scanpy, scvi-tools, celltypist, anndata, pandas, numpy, seaborn, matplotlib.
- External tools: cell_type_mapper (Allen Institute).

## Citations
- Scanpy: Wolf, F. A. et al. Scanpy: large-scale single-cell gene expression data analysis. Genome Biol. 19, 15 (2018).
- scvi-tools: Gayoso, A. et al. A Python library for probabilistic analysis of single-cell omics data. Nat. Methods 19, 445-454 (2022).
- MapMyCells: Yao, Z. et al. A high-resolution transcriptomic and spatial atlas of cell types in the whole mouse brain. Nature 624, 316-325 (2023). PMID: 38092916.
- AnnData documentation: Concatenation. https://anndata.readthedocs.io/en/latest/concatenation.html

## Setup Instructions
1. Run notebook 00 first to generate scAR-corrected sample `.h5ad` files.
2. Install dependencies: `conda env create -f environment.yml` or `pip install -r requirements.txt`.
3. Ensure corrected sample files are present in `Analysis/scAR/h5ad/`.
4. Run cells sequentially. MapMyCells requires pre-downloaded marker files (see terminal commands).

## QC Parameters
- **Min genes per cell**: 100.
- **Mitochondrial filter**: <5% of counts (`pct_counts_mt < 5`).
- **Doublet detection**: SCVI + SOLO (high-confidence doublets removed).
- **HVG selection**: 3000 genes (seurat_v3 flavor).

## Notes
- Paths are configurable via `BASE_DIR` and `SCAR_H5AD_DIR`.
- For reproducibility, random seeds are set.
- Ambient correction is assumed complete before this notebook starts.

## Troubleshooting
- **Corrected files missing**: Re-run notebook 00 or verify `Analysis/scAR/h5ad/`.
- **MapMyCells missing**: Install `cell_type_mapper` from GitHub (`pip install git+https://github.com/AllenInstitute/cell_type_mapper.git`). Pre-download taxonomy files to `CELL_TYPE_MAPPER_WMB_DIR`.
- **HDF5/H5AD errors**: Check file paths and permissions.
- **Memory issues**: Reduce `n_top_genes` or `n_neighbors` in Scanpy functions for large datasets.
- **UMAP plots not saving**: Ensure `UMAP_DIR` exists and is writable.

## Version History
- v1.0 (2024-03-25): Initial production notebook built around filtered 10x inputs.
- v1.1 (2026-03-25): Relaxed QC threshold to `min_genes=100` for immune retention after ambient RNA filter.
- v1.2 (2026-04-01): Converted ingest to start from notebook 00 scAR-corrected `.h5ad` files and removed CellBender from the workflow.
- v1.3 (2026-04-01): Standardized AnnData output names to the `adata_scar_*` convention.
- v1.4 (2026-04-01): Added notebook-tagged output naming (`nb01`) for all exported artifacts and exported count-sum CSV tables.
- v1.5 (2026-04-02): Added reusable barcode export helpers, full-cell barcode table export, and an annotated HVG-filtered AnnData output.
- v1.6 (2026-04-07): Replaced Scanorama integration with sc.concat-based integration and PCA compatibility embedding.

### Environment Note (MapMyCells)

To keep this notebook portable across machines, set `CELL_TYPE_MAPPER_WMB_DIR` to your local taxonomy folder before running the mapping cell.

Example folder contents expected:
- `mouse_markers_230821.json`
- `precomputed_stats_ABC_revision_230821.h5`

# Table of Contents
1. [Imports and Setup](#imports)
2. [Data Preparation](#data-prep)
3. [Preprocessing and QC](#preprocessing)
4. [Integration](#integration)
5. [Cell Type Mapping](#mapping)
6. [Visualization and Analysis](#analysis)
7. [Subsetting and Exports](#subsetting)
8. [References](#references)

### Description
This notebook ingests the scAR-corrected outputs produced upstream in notebook 00 for the brain snRNA-seq study comparing *Candida albicans*-infected mice and mock controls.

project path: /media/drive_c/Project_Brain_snRNAseq
corrected input path: /media/drive_c/Project_Brain_snRNAseq/Analysis/scAR/h5ad

In [ ]:
# Consolidated imports (PEP8: stdlib, third-party, local)
import os
import time
import json
import logging
import textwrap
from pathlib import Path
from typing import Optional, Dict, List

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scvi
import torch
import scanorama

import seaborn as sns
import matplotlib.pyplot as plt
import umap
import celltypist as ctyp
from celltypist import models
import statannotations as stANN

import scipy.stats as stats
from scipy.stats import ttest_ind, sem, gaussian_kde

# Local imports will be added after BASE_DIR is set

In [ ]:
# Version logging for reproducibility
import sys
print(f"Python version: {sys.version}")
print(f"Scanpy: {sc.__version__}")
print(f"scvi: {scvi.__version__}")
print(f"Celltypist: {ctyp.__version__}")
print(f"Anndata: {ad.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")
import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

In [ ]:
# Configuration: Set base directory, corrected input directory, output naming, and random seeds
NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name != "Notebooks":
    NOTEBOOKS_DIR = (NOTEBOOKS_DIR / "Notebooks").resolve()
ANALYSIS_DIR = NOTEBOOKS_DIR.parent.resolve()
BASE_DIR = ANALYSIS_DIR.parent.resolve()
ADATA_DIR = ANALYSIS_DIR / "adatas"
MAPPING_DIR = ANALYSIS_DIR / "Mapping"
RESULTS_DIR = ANALYSIS_DIR / "Results"
UMAP_DIR = RESULTS_DIR / "UMAP"
MICROGLIA_DIR = ANALYSIS_DIR / "Microglia_analysis"
ASTRO_DIR = ANALYSIS_DIR / "Astrocyte_analysis"
TABLE_DIR = RESULTS_DIR / "Tables"
SCAR_H5AD_DIR = ANALYSIS_DIR / "scAR" / "h5ad"
QC_PLOT_DIR = ANALYSIS_DIR / "QC_plots"

# Notebook-aware output naming (all exported files include source notebook tag)
NOTEBOOK_TAG = "nb01"
ADATA_STEM = f"adata_scar_{NOTEBOOK_TAG}"

ADATA_INTEGRATED_PATH = ADATA_DIR / f"{ADATA_STEM}_integrated.h5ad"
ADATA_GENEIDS_QUERY_PATH = ADATA_DIR / f"{ADATA_STEM}_geneids_query.h5ad"
ADATA_CLASS_FILTERED_BASE = ADATA_DIR / f"{ADATA_STEM}_class_filtered"
ADATA_ANNOTATED_HVG_PATH = ADATA_DIR / f"{ADATA_STEM}_annotated_hvg.h5ad"

SAMPLE_PATHS_CSV = NOTEBOOKS_DIR / f"sample_paths_{NOTEBOOK_TAG}.csv"
MAPPING_CSV_PATH = MAPPING_DIR / f"mapping_output_{NOTEBOOK_TAG}.csv"
MAPPING_JSON_PATH = MAPPING_DIR / f"mapping_output_{NOTEBOOK_TAG}.json"

UMAP_CLASS_BASE = UMAP_DIR / f"UMAP_Cell_Class_{NOTEBOOK_TAG}"
UMAP_CLASS_COUNTS_PATH = UMAP_DIR / f"UMAP_Cell_Class_counts_{NOTEBOOK_TAG}.svg"

CLASS_COUNTS_BY_SAMPLE_CSV = TABLE_DIR / f"cell_counts_class_by_sample_{NOTEBOOK_TAG}.csv"
CLASS_COUNTS_SUM_CSV = TABLE_DIR / f"cell_counts_class_sum_{NOTEBOOK_TAG}.csv"
CLASS_STATS_CSV = TABLE_DIR / f"cell_counts_class_stats_{NOTEBOOK_TAG}.csv"

MICROGLIA_BARCODES_CSV = MICROGLIA_DIR / f"microglia_cell_barcodes_{NOTEBOOK_TAG}.csv"
ASTRO_BARCODES_CSV = ASTRO_DIR / f"astrocyte_cell_barcodes_{NOTEBOOK_TAG}.csv"
ALL_CELL_BARCODES_CSV = TABLE_DIR / f"all_cell_barcodes_{NOTEBOOK_TAG}.csv"

# MapMyCells taxonomy config (override with env var CELL_TYPE_MAPPER_WMB_DIR if needed).
CELL_TYPE_MAPPER_WMB_DIR = Path(
    os.environ.get("CELL_TYPE_MAPPER_WMB_DIR", "/home/jmk/cell_type_mapper/Taxonomies/WMB")
)
MARKER_LOOKUP_PATH = CELL_TYPE_MAPPER_WMB_DIR / "mouse_markers_230821.json"
PRECOMPUTED_STATS_PATH = CELL_TYPE_MAPPER_WMB_DIR / "precomputed_stats_ABC_revision_230821.h5"

# Use a conservative default that scales with host CPUs.
N_MMC_PROCESSORS = max(1, min(16, (os.cpu_count() or 4) - 1))

# Set random seeds
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
import random
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
scvi.settings.seed = RANDOM_STATE

# Configure Scanpy settings
sc.settings.set_figure_params(dpi=100)
sc.settings.verbosity = 2
for d in [ANALYSIS_DIR, ADATA_DIR, MAPPING_DIR, RESULTS_DIR, UMAP_DIR, MICROGLIA_DIR, ASTRO_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook tag: {NOTEBOOK_TAG}")
print(f"Integrated AnnData path: {ADATA_INTEGRATED_PATH}")
print(f"GeneID query AnnData path: {ADATA_GENEIDS_QUERY_PATH}")
print(f"Class-filtered AnnData base: {ADATA_CLASS_FILTERED_BASE}")
print(f"Annotated HVG AnnData path: {ADATA_ANNOTATED_HVG_PATH}")
print(f"All barcodes CSV path: {ALL_CELL_BARCODES_CSV}")
print(f"MapMyCells CSV: {MAPPING_CSV_PATH}")
print(f"MapMyCells JSON: {MAPPING_JSON_PATH}")
print(f"MapMyCells taxonomy dir: {CELL_TYPE_MAPPER_WMB_DIR}")
print(f"Marker lookup exists: {MARKER_LOOKUP_PATH.exists()}")
print(f"Precomputed stats exists: {PRECOMPUTED_STATS_PATH.exists()}")
print(f"MapMyCells processors: {N_MMC_PROCESSORS}")

In [ ]:
# Create a DataFrame of scAR-corrected sample h5ad files
h5ad_files = sorted(SCAR_H5AD_DIR.glob('*_scar_corrected.h5ad'))

if not h5ad_files:
    raise FileNotFoundError(f'No corrected h5ad files found in: {SCAR_H5AD_DIR}')

df = pd.DataFrame(
    {
        'Sample': [p.name.replace('_scar_corrected.h5ad', '') for p in h5ad_files],
        'Path': [str(p) for p in h5ad_files],
    }
).sort_values('Sample').reset_index(drop=True)

df['Condition'] = df['Sample'].str.startswith('OG-').map({True: 'OG', False: 'Mock'})
df.to_csv(SAMPLE_PATHS_CSV, index=False)

print(f"Saved sample path table: {SAMPLE_PATHS_CSV}")
print(df)

# Preparing AnnData for analysis from scAR-corrected inputs

In [ ]:
# Load shared preprocessing helpers from Analysis/utils.py
import importlib.util

UTILS_PATH = ANALYSIS_DIR / "utils.py"
if not UTILS_PATH.exists():
    raise FileNotFoundError(f"Missing utils.py at: {UTILS_PATH}")

# Load module with error checking
spec = importlib.util.spec_from_file_location("analysis_utils", UTILS_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not create import spec for: {UTILS_PATH}")

# Initial load
analysis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis_utils)

# Reload to capture any updates to function signatures
spec = importlib.util.spec_from_file_location("analysis_utils", UTILS_PATH)
if spec and spec.loader:
    analysis_utils = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(analysis_utils)

# Extract and export module functions
pp = analysis_utils.pp
plot_embedding = analysis_utils.plot_embedding

print(f"Loaded helper module: {UTILS_PATH}")
print(f"pp available: {callable(pp)}")
print(f"plot_embedding available: {callable(plot_embedding)}")
print("Module reloaded for latest function signatures")

In [ ]:
# plot_embedding helper is loaded from utils.py above

In [ ]:
# Create an empty list to store AnnaData objects
datas = []

## Import, Filter, Concat, and Var/groups addition

In [ ]:
# Loop through each corrected sample AnnData path

for index, row in df.iterrows():
    h5_path = row['Path']
    sample_id = row['Sample']
    print(f"Processing row {index}: h5_path='{h5_path}', sample_id='{sample_id}'")
    
    start_time = time.time()
    adata = pp(h5_path, sample_id, min_genes=100, qc_plot_dir=QC_PLOT_DIR)
    end_time = time.time()
    
    elapsed_time = end_time - start_time
    print(f"Processing time for row {index}: {elapsed_time:.2f} seconds")
    
    datas.append(adata)

In [ ]:
# Concatenate the list of AnnData objects into a single AnnData object
adata = sc.concat(datas, join ='outer', uns_merge = 'same')
adata



In [ ]:
# Preserve .var[gene_ids]

# Define which columns you want to keep
keep_cols = ["gene_ids"]

# Grab .var from each dataset in the list and subset to columns we want (if they exist)
all_var = [
    x.var.loc[:, [c for c in keep_cols if c in x.var.columns]]
    for x in datas
]

# Concatenate and drop duplicate gene names
all_var = pd.concat(all_var, join="outer")
all_var = all_var[~all_var.index.duplicated()]

# Subset to adata genes
adata.var = all_var.loc[adata.var_names]



In [ ]:
# Creating a treatment group obs for downstream analysis

# Define sample groups
naive_samples = ['Mock-1', 'Mock-2', 'Mock-3']
infected_samples = ['OG-1', 'OG-2', 'OG-3']

# Create a new column 'treatment' in adata.obs
adata.obs['treatment'] = adata.obs['Sample'].apply(
    lambda x: 'Naive' if x in naive_samples else ('Infected' if x in infected_samples else 'Unknown')
)

# Convert 'treatment' column to categorical type
adata.obs['treatment'] = adata.obs['treatment'].astype('category')

## Scanorama for UMAP projection

In [ ]:
# Run scanorama integration on the list of AnnData objects
scanorama.integrate_scanpy(datas)

# Get all the integrated matrices
scanorama_int = [ad.obsm['X_scanorama'] for ad in datas]

# make into one matrix
adata.obsm["Scanorama"] = np.concatenate(scanorama_int)

In [ ]:
adata

# Save Load Point: `adata_scar_integrated.h5ad`

In [ ]:
# Save integrated object with raw counts in X for downstream compatibility
adata.X = adata.layers['counts']
adata.write_h5ad(ADATA_INTEGRATED_PATH)
print(f"Saved integrated AnnData: {ADATA_INTEGRATED_PATH}")

In [ ]:
# Read back integrated AnnData
adata = ad.read_h5ad(ADATA_INTEGRATED_PATH)
print(f"Loaded: {ADATA_INTEGRATED_PATH}")

In [ ]:
adata

# MapMyCells

Publication: A high-resolution transcriptomic and spatial atlas of cell types in the whole mouse brain (PMID: 38092916)
Source: https://github.com/AllenInstitute/cell_type_mapper


In [ ]:
# Prepare a MapMyCells query object while avoiding a full in-memory adata copy
if 'gene_ids' not in adata.var.columns:
    raise ValueError("Missing 'gene_ids' in adata.var")

var_query = adata.var[['gene_ids']].copy()
var_query['gene_symbol'] = adata.var_names.astype(str)
var_query['gene_ids'] = var_query['gene_ids'].astype(str)

if var_query['gene_ids'].duplicated().any():
    raise ValueError("Duplicate gene_ids found; MapMyCells query index must be unique.")

var_query = var_query.set_index('gene_ids')

# Keep only what MapMyCells needs; this avoids duplicating obsm/varm/layers/uns from adata.
adata_query = ad.AnnData(X=adata.X, obs=adata.obs.copy(deep=False), var=var_query)
adata_query.write_h5ad(ADATA_GENEIDS_QUERY_PATH)

print(f"Saved gene_ids query AnnData: {ADATA_GENEIDS_QUERY_PATH}")
print(f"Index name: {adata_query.var.index.name} | Unique: {adata_query.var.index.is_unique}")

# Free temporary query objects immediately after writing to reduce peak memory.
del adata_query, var_query
import gc
gc.collect()

In [ ]:
# Optional quick sanity check of query index (read in backed mode; no full object in memory)
adata_query_check = ad.read_h5ad(ADATA_GENEIDS_QUERY_PATH, backed='r')
print(adata_query_check.var_names[:5].tolist())
adata_query_check.file.close()
del adata_query_check

### Preparing to run cell type mapper...

Note:
Cells are mapped with the original whole mouse brain atlas taxonomy from using MapMyCells

(https://portal.brain-map.org/atlases-and-data/bkp/mapmycells)




In [ ]:
# Run MapMyCells via subprocess so this works in a standard Python notebook cell.
import subprocess
import sys

if not MARKER_LOOKUP_PATH.exists() or not PRECOMPUTED_STATS_PATH.exists():
    raise FileNotFoundError(
        "Missing MapMyCells taxonomy files. Set CELL_TYPE_MAPPER_WMB_DIR or update paths in config cell.\n"
        f"Expected marker file: {MARKER_LOOKUP_PATH}\n"
        f"Expected precomputed stats: {PRECOMPUTED_STATS_PATH}"
    )

cmd = [
    sys.executable,
    "-m",
    "cell_type_mapper.cli.from_specified_markers",
    "--query_path", str(ADATA_GENEIDS_QUERY_PATH),
    "--extended_result_path", str(MAPPING_JSON_PATH),
    "--csv_result_path", str(MAPPING_CSV_PATH),
    "--drop_level", "CCN20230722_SUPT",
    "--cloud_safe", "False",
    "--query_markers.serialized_lookup", str(MARKER_LOOKUP_PATH),
    "--precomputed_stats.path", str(PRECOMPUTED_STATS_PATH),
    "--type_assignment.normalization", "raw",
    "--type_assignment.n_processors", str(N_MMC_PROCESSORS),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Mapping Outputs

> Use the CSV as the canonical annotation table for downstream analysis and attach it to `adata.obs` once.
The JSON file is kept as compact taxonomy metadata for optional inspection, not as the primary working data structure.

In [ ]:
# Load MapMyCells outputs once and standardize the downstream annotation table
mapping_csv = MAPPING_CSV_PATH
mapping_json = MAPPING_JSON_PATH

if not mapping_csv.exists():
    candidates = sorted(MAPPING_DIR.glob(f"*{NOTEBOOK_TAG}*.csv")) or sorted(MAPPING_DIR.glob("*.csv"))
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No mapping CSV found in {MAPPING_DIR}. Run MapMyCells CLI block first and ensure output path is correct."
        )
    mapping_csv = candidates[0]

try:
    celltype_table = pd.read_csv(mapping_csv, comment='#')
except pd.errors.EmptyDataError:
    raise ValueError(f"Mapping CSV exists but is empty: {mapping_csv}")
except pd.errors.ParserError as exc:
    raise ValueError(f"Parser error reading mapping CSV {mapping_csv}: {exc}")

required_cols = [
    'cell_id',
    'class_name',
    'subclass_name',
    'supertype_name',
    'cluster_name',
    'cluster_alias',
]
missing_cols = [col for col in required_cols if col not in celltype_table.columns]
if missing_cols:
    raise ValueError(f"Mapping CSV is missing expected columns: {missing_cols}")

celltype_table = celltype_table.drop_duplicates(subset='cell_id').copy()
celltype_table_indexed = celltype_table.set_index('cell_id')
probability_cols = [col for col in celltype_table.columns if col.endswith('_bootstrapping_probability')]
mapping_levels = {
    'class': ('class_name', 'class_bootstrapping_probability'),
    'subclass': ('subclass_name', 'subclass_bootstrapping_probability'),
    'supertype': ('supertype_name', 'supertype_bootstrapping_probability'),
    'cluster': ('cluster_name', 'cluster_bootstrapping_probability'),
}

# Preserve the historical variable name for compatibility with any untouched cells.
csv_results = celltype_table.copy()

print(f"Loaded mapping CSV from {mapping_csv}")
print(f"Rows: {len(celltype_table):,} | Columns: {len(celltype_table.columns)}")
print(f"Probability columns: {probability_cols}")
display(celltype_table.head())

In [ ]:
list(csv_results.columns)

## Attach Mapping to AnnData

> Attach the CSV-derived annotations once, then use `adata.obs` as the single source of truth for plotting and subsetting.

In [ ]:
# Merge MapMyCells annotations into adata.obs once and derive broad plotting classes
def to_broad_class(name: str) -> str:
    if pd.isna(name):
        return 'Unmapped'
    name = str(name).strip()
    broad_labels = [
        'Glut', 'GABA', 'Dopa', 'Sero', 'Astro', 'OPC', 'OEC', 'Vascular', 'Immune',
    ]
    for label in broad_labels:
        if label in name:
            return label
    return name if name else 'Unmapped'

annotation_cols = [col for col in celltype_table.columns if col != 'cell_id']
existing_annotation_cols = [col for col in annotation_cols if col in adata.obs.columns]
if existing_annotation_cols:
    adata.obs = adata.obs.drop(columns=existing_annotation_cols)

adata.obs = adata.obs.join(celltype_table_indexed, how='left')

if 'Condition' not in adata.obs.columns and 'Sample' in adata.obs.columns:
    adata.obs['Condition'] = np.where(
        adata.obs['Sample'].astype(str).str.contains('^OG-', regex=True, na=False),
        'OG',
        'Mock',
    )

adata.obs['class_broad'] = adata.obs['class_name'].map(to_broad_class)
adata.obs['class_broad'] = pd.Categorical(
    adata.obs['class_broad'],
    categories=['Glut', 'GABA', 'Dopa', 'Sero', 'Astro', 'OPC', 'OEC', 'Vascular', 'Immune', 'Unmapped'],
    ordered=False,
)

annotation_preview_cols = [
    'class_name', 'subclass_name', 'supertype_name', 'cluster_name', 'cluster_alias', 'class_broad',
]
display(adata.obs[annotation_preview_cols].head())
print('Annotated cells:', int(adata.obs['class_name'].notna().sum()))
print('Broad classes:', adata.obs['class_broad'].value_counts(dropna=False).to_dict())

In [ ]:
adata.obs

## Optional: Taxonomy Metadata

> The JSON is useful for auditing the returned taxonomy structure, but downstream analysis should use the annotations already attached to `adata.obs`.

In [ ]:
# Read taxonomy metadata once for compact inspection
if not mapping_json.exists():
    raise FileNotFoundError(f"Mapping JSON not found: {mapping_json}")

with open(mapping_json, 'r') as src:
    json_results = json.load(src)

taxonomy_tree = json_results['taxonomy_tree']
taxonomy_hierarchy = taxonomy_tree.get('hierarchy', [])
taxonomy_summary = pd.DataFrame(
    [
        {'level': level, 'n_nodes': len(taxonomy_tree.get(level, {}))}
        for level in taxonomy_hierarchy
    ]
)

print(f"Loaded taxonomy metadata from {mapping_json}")
print(f"Mapped cells in JSON payload: {len(json_results.get('results', [])):,}")
display(taxonomy_summary)

## Cell Type UMAP

Build the broad class UMAP from `adata.obs` and save the filtered annotated object for downstream reuse.

In [ ]:
# Condensed helper functions for post-mapping visualization and summaries
def build_barcode_to_label(obs_df: pd.DataFrame, label_col: str) -> dict[str, str]:
    labels = obs_df[label_col].astype(str)
    excluded = {'nan', 'none', 'unmapped', ''}
    return {
        str(barcode): label
        for barcode, label in labels.items()
        if label.strip().lower() not in excluded
    }


def add_group_column(obs_df: pd.DataFrame) -> pd.DataFrame:
    out = obs_df.copy()

    # Prefer Sample-derived grouping to avoid contaminated/misaligned Condition columns.
    if 'Sample' in out.columns:
        out['group'] = np.where(
            out['Sample'].astype(str).str.contains('^OG-', regex=True, na=False),
            'OG',
            'Mock',
        )
    elif 'Condition' in out.columns:
        out['group'] = out['Condition'].astype(str)
    else:
        raise ValueError("Need either 'Sample' or 'Condition' to derive groups.")

    out = out[out['group'].isin(['Mock', 'OG'])].copy()
    return out


def plot_grouped_counts(
    adata_obj: ad.AnnData,
    category_col: str,
    save_path: Path,
    title: str,
    xlabel: str,
    subset_mask=None,
    rotate_xticks: bool = False,
    figsize=(12, 6),
    by_sample_csv_path: Path | None = None,
    sum_csv_path: Path | None = None,
    stats_csv_path: Path | None = None,
    figure_caption: str | None = None,
    caption_fontsize: int = 16,
    caption_char_width: int = 85,
    caption_xy: tuple = (0.05, -0.10),
    legend_title: str = 'Group',
):
    import textwrap

    obs_df = adata_obj.obs.copy() if subset_mask is None else adata_obj.obs.loc[subset_mask].copy()
    obs_df = add_group_column(obs_df)

    required_cols = [category_col, 'Sample', 'group']
    missing_cols = [col for col in required_cols if col not in obs_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns for grouped count plot: {missing_cols}")

    obs_df = obs_df.dropna(subset=[category_col]).copy()
    obs_df[category_col] = obs_df[category_col].astype(str)

    count_df = (
        obs_df.groupby(['Sample', category_col, 'group'])
        .size()
        .reset_index(name='count')
    )

    sum_df = (
        count_df.groupby([category_col, 'group'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'count_sum'})
        .sort_values([category_col, 'group'])
    )

    stats_rows = []
    for category in sorted(count_df[category_col].unique()):
        sub = count_df[count_df[category_col] == category]
        mock_vals = sub.loc[sub['group'] == 'Mock', 'count'].to_numpy()
        og_vals = sub.loc[sub['group'] == 'OG', 'count'].to_numpy()

        if len(mock_vals) > 0 and len(og_vals) > 0:
            _, p_val = ttest_ind(mock_vals, og_vals, equal_var=False)
        else:
            p_val = np.nan

        stats_rows.append({'category': category, 'p_value': p_val})

    stats_df = pd.DataFrame(stats_rows).sort_values('p_value', na_position='last')

    if by_sample_csv_path is not None:
        count_df.to_csv(by_sample_csv_path, index=False)
        print(f"Saved counts by sample: {by_sample_csv_path}")
    if sum_csv_path is not None:
        sum_df.to_csv(sum_csv_path, index=False)
        print(f"Saved count sums: {sum_csv_path}")
    if stats_csv_path is not None:
        stats_df.to_csv(stats_csv_path, index=False)
        print(f"Saved count stats: {stats_csv_path}")

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=count_df,
        x=category_col,
        y='count',
        hue='group',
        errorbar='se',
        capsize=0.1,
        errwidth=1.5,
        palette='Set2',
        ax=ax,
    )
    sns.stripplot(
        data=count_df,
        x=category_col,
        y='count',
        hue='group',
        dodge=True,
        color='black',
        alpha=0.7,
        ax=ax,
    )

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title=legend_title)
    ax.set_ylabel('Cell Count')
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    if rotate_xticks:
        ax.tick_params(axis='x', labelrotation=45)
        for tick in ax.get_xticklabels():
            tick.set_ha('right')

    if figure_caption:
        wrapped_caption = '\n'.join(
            textwrap.wrap(str(figure_caption), width=max(20, int(caption_char_width)))
        )
        fig.text(
            caption_xy[0],
            caption_xy[1],
            wrapped_caption,
            fontsize=caption_fontsize,
            ha='left',
            va='top',
            transform=fig.transFigure,
        )

    fig.tight_layout()
    fig.savefig(save_path, format='svg')
    plt.show()

    return {'stats_df': stats_df, 'count_df': count_df, 'sum_df': sum_df}

In [ ]:
# Build broad class labels directly from the annotations attached to adata.obs
if 'class_broad' not in adata.obs.columns:
    raise ValueError("Run the annotation-merge cell first so 'class_broad' exists in adata.obs.")

barcode_to_label = build_barcode_to_label(adata.obs, 'class_broad')
preferred_order = ['Glut', 'GABA', 'Dopa', 'Sero', 'Astro', 'OPC', 'OEC', 'Vascular', 'Immune']
label_order = [label for label in preferred_order if label in set(barcode_to_label.values())]
for label in sorted(set(barcode_to_label.values())):
    if label not in label_order:
        label_order.append(label)

print(f"Cells with broad labels for plotting: {len(barcode_to_label):,} / {adata.n_obs:,}")
print('label_order:', label_order)
display(adata.obs[['class_name', 'class_broad', 'subclass_name', 'cluster_name']].head())

In [ ]:
# Plot class-level UMAP from the annotations already attached to adata.obs
# Recompute here so UMAP geometry reflects the tuned settings below.
# This is signature-safe across helper versions and prefers Scanorama when present.
import inspect

plot_sig = inspect.signature(plot_embedding)
plot_kwargs = dict(
    barcode_to_label=barcode_to_label,
    save_path=UMAP_CLASS_BASE,
    filtered_adata=ADATA_CLASS_FILTERED_BASE,
    label_order=label_order,
    raw_data=adata,
    palette_name='tab10',
    n_neighbors=450,
    min_dist=1.6,
    spread=0.9,
    save_format='svg',
    use_cached=False,
    force_recompute=True,
)

if 'integration_key' in plot_sig.parameters:
    plot_kwargs['integration_key'] = 'Scanorama'

if 'figure_caption' in plot_sig.parameters:
    plot_kwargs['figure_caption'] = (
        'Figure. UMAP projection of integrated brain snRNA-seq data '
        'colored by broad cell class after Scanorama integration.'
    )
if 'caption_fontsize' in plot_sig.parameters:
    plot_kwargs['caption_fontsize'] = 16
if 'caption_char_width' in plot_sig.parameters:
    plot_kwargs['caption_char_width'] = 85
if 'caption_xy' in plot_sig.parameters:
    plot_kwargs['caption_xy'] = (0.15, 0.08)

class_filtered_path = plot_embedding(**plot_kwargs)

print('Saved class UMAP figure:', UMAP_CLASS_BASE.with_suffix('.svg'))
print('Saved filtered annotated AnnData:', class_filtered_path)

## Cell Type Barcode Export

Export barcodes for selected cell types from `adata.obs` (no AnnData subset copy).

In [ ]:
# Reusable barcode export utilities for downstream analysis handoff
def export_celltype_barcodes(
    adata_obj: ad.AnnData,
    query: str,
    output_csv: Path,
    label_col: str = 'cluster_name',
    match_mode: str = 'contains',
    case: bool = False,
    barcode_col: str = 'cell_barcode',
) -> pd.DataFrame:
    """Export barcodes for a selected cell type label.

    Parameters
    ----------
    adata_obj : ad.AnnData
        Annotated AnnData object with labels in `obs`.
    query : str
        Text query used to select rows in `label_col`.
    output_csv : Path
        Destination CSV path for exported barcodes.
    label_col : str
        obs column used for matching (default: `cluster_name`).
    match_mode : str
        Matching strategy: `contains` or `equals`.
    case : bool
        If False, matching is case-insensitive.
    barcode_col : str
        Output CSV column name for barcode values.

    Returns
    -------
    pd.DataFrame
        One-column DataFrame of exported barcodes.
    """
    if label_col not in adata_obj.obs.columns:
        raise ValueError(f"Column '{label_col}' not found in adata.obs.")

    labels = adata_obj.obs[label_col].astype(str)
    if match_mode == 'contains':
        mask = labels.str.contains(query, case=case, na=False)
    elif match_mode == 'equals':
        if case:
            mask = labels == query
        else:
            mask = labels.str.lower() == str(query).lower()
    else:
        raise ValueError("match_mode must be 'contains' or 'equals'.")

    barcodes = adata_obj.obs.index[mask].astype(str)
    out_df = pd.DataFrame({barcode_col: barcodes})
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(output_csv, index=False)

    print(f"Saved {query} barcodes: {output_csv}")
    print(f"Rows exported: {len(out_df):,}")
    return out_df


def export_all_barcodes(
    adata_obj: ad.AnnData,
    output_csv: Path,
    barcode_col: str = 'cell_barcode',
) -> pd.DataFrame:
    """Export all cell barcodes with available annotation context.

    Returns a table keyed by cell barcode and includes common metadata
    columns if they exist in `adata.obs`.
    """
    metadata_cols = [
        'Sample',
        'Condition',
        'class_name',
        'class_broad',
        'subclass_name',
        'cluster_name',
        'cluster_alias',
    ]
    keep_cols = [col for col in metadata_cols if col in adata_obj.obs.columns]

    out_df = adata_obj.obs.loc[:, keep_cols].copy()
    out_df.insert(0, barcode_col, adata_obj.obs_names.astype(str))
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(output_csv, index=False)

    print(f"Saved all barcodes table: {output_csv}")
    print(f"Rows exported: {len(out_df):,}")
    return out_df

In [ ]:
# Microglia Barcodes
microglia_barcodes = export_celltype_barcodes(
    adata_obj=adata,
    query='Microglia',
    output_csv=MICROGLIA_BARCODES_CSV,
    label_col='cluster_name',
    match_mode='contains',
)


In [ ]:
# Astrocyte Barcodes
astrocyte_barcodes = export_celltype_barcodes(
    adata_obj=adata,
    query='astro',
    output_csv=ASTRO_BARCODES_CSV,
    label_col='cluster_name',
    match_mode='contains',
)

In [ ]:
# Save annotated HVG-filtered AnnData for downstream analyses (inline one-time block)
if adata.n_obs == 0 or adata.n_vars == 0:
    raise ValueError('Cannot compute HVGs on an empty AnnData object.')

source_layer = 'counts'
x_source = adata.layers[source_layer] if source_layer in adata.layers else adata.X
hvg_work = ad.AnnData(
    X=x_source,
    obs=adata.obs.copy(deep=False),
    var=adata.var.copy(deep=True),
)

hvg_kwargs = {'n_top_genes': 3000, 'flavor': 'seurat_v3', 'subset': False}
if 'Sample' in hvg_work.obs.columns:
    hvg_kwargs['batch_key'] = 'Sample'
sc.pp.highly_variable_genes(hvg_work, **hvg_kwargs)

if 'highly_variable' not in hvg_work.var.columns:
    raise ValueError("HVG computation failed: 'highly_variable' column not found.")

hvg_mask = hvg_work.var['highly_variable'].astype(bool).to_numpy()
n_hvg_selected = int(hvg_mask.sum())
if n_hvg_selected == 0:
    raise ValueError('HVG computation returned 0 genes.')

adata_hvg = adata[:, hvg_mask].copy()
adata_hvg.uns['hvg_n_top_genes_requested'] = 3000
adata_hvg.uns['hvg_n_genes_selected'] = n_hvg_selected
adata_hvg.uns['hvg_source_layer'] = source_layer if source_layer in adata.layers else 'X'
adata_hvg.uns['source_notebook'] = '01_Ingest.ipynb'

ADATA_ANNOTATED_HVG_PATH.parent.mkdir(parents=True, exist_ok=True)
adata_hvg.write_h5ad(ADATA_ANNOTATED_HVG_PATH)
print(f"Saved annotated HVG AnnData: {ADATA_ANNOTATED_HVG_PATH}")
print(f"Shape: {adata_hvg.n_obs:,} cells x {adata_hvg.n_vars:,} genes")

del hvg_work

In [ ]:
# Export all cell barcodes with cell/group annotation context
all_cell_barcodes = export_all_barcodes(
    adata_obj=adata,
    output_csv=ALL_CELL_BARCODES_CSV,
)

In [ ]:
# Pipeline completion summary
print("=" * 60)
print("PIPELINE COMPLETE - Brain snRNAseq Ingestion")
print("=" * 60)

print(f"\nFinal dataset metrics:")
print(f"  Total cells: {adata.n_obs:,}")
print(f"  Total genes: {adata.n_vars:,}")
print(f"  Samples: {adata.obs['Sample'].nunique()}")

# Count exported subsets
microglia_count = len(pd.read_csv(MICROGLIA_BARCODES_CSV))
astrocyte_count = len(pd.read_csv(ASTRO_BARCODES_CSV))
all_barcodes_count = len(pd.read_csv(ALL_CELL_BARCODES_CSV))

print(f"\nExported barcode tables:")
print(f"  All cells: {all_barcodes_count}")
print(f"  Microglia: {microglia_count}")
print(f"  Astrocytes: {astrocyte_count}")

print(f"\nOutput files saved to:")
print(f"  Processed AnnData: {ADATA_INTEGRATED_PATH}")
print(f"  MapMyCells query AnnData: {ADATA_GENEIDS_QUERY_PATH}")
print(f"  Class-filtered AnnData base: {ADATA_CLASS_FILTERED_BASE}")
print(f"  Annotated HVG AnnData: {ADATA_ANNOTATED_HVG_PATH}")
print(f"  Cell type mappings CSV: {MAPPING_CSV_PATH}")
print(f"  Cell type mappings JSON: {MAPPING_JSON_PATH}")
print(f"  Class UMAP plot: {UMAP_CLASS_BASE.with_suffix('.svg')}")
print(f"  All cell barcodes: {ALL_CELL_BARCODES_CSV}")
print(f"  Microglia barcodes: {MICROGLIA_BARCODES_CSV}")
print(f"  Astrocyte barcodes: {ASTRO_BARCODES_CSV}")
print("\n" + "=" * 60)

# References
- Wolf, F. A. et al. Scanpy: large-scale single-cell gene expression data analysis. Genome Biol. 19, 15 (2018).
- Gayoso, A. et al. A Python library for probabilistic analysis of single-cell omics data. Nat. Methods 19, 445-454 (2022).
- Yao, Z. et al. A high-resolution transcriptomic and spatial atlas of cell types in the whole mouse brain. Nature 624, 316-325 (2023). PMID: 38092916.
- AnnData documentation: Concatenation. https://anndata.readthedocs.io/en/latest/concatenation.html
- Dominguez Conde, C. et al. Cross-tissue immune cell analysis reveals tissue-specific features in humans. Science 376, eabl5197 (2022). (For celltypist)